# 07 — Validate, launch and resume the long run

This notebook continues the fresh **AHR v2** sequence. It launches the authorized 12-hour, 9×9 AHR self-play run in detached tmux. The old runs remain separate. The network starts from fresh weights; no teacher labels are used.

**Inputs:** notebooks 00–06, their validation receipts, `config/ahr_v2_training.json`, GPU 1. **Outputs:** the detached worker notebook, launch receipt and progress under `reports/ahr_v2/`; full resumable state under `checkpoints/<run_id>/`. Normal Run All takes less than a minute and shows readiness. Training takes the configured active compute budget plus bounded development reviews.

**Run order:** 00–03 foundations → 04 self-play and training → 05 evaluation and progress → 06 play and watch → 07 launch. Open 05 to inspect progress and 06 to play against a pinned saved model. A saved model is usable while training continues; its strength must be measured separately.

**Checkpoint and resume:** use the same configuration and run ID; the loader checks source, configuration and checkpoint hashes and resumes the recorded game, search, replay, optimizer and RNG state. It never treats a corrupt checkpoint as a fresh run. The latest two valid checkpoints and the best recorded training-loss checkpoint are retained. Only one writer may own a run.

**Connection independence:** tmux belongs to the GPU instance, so disconnecting the notebook client does not stop training. Turning off or deleting the instance stops compute. This workspace is on the boot disk; the instance deletion policy remains unverified. Off-instance backup is deferred at the user's request until their storage destination is ready.

Periodic development reviews run about every two hours and a larger development review runs at the end. Review games are excluded from training, and review time is excluded from the training active-time budget. A diagnostic error is recorded separately without claiming a match result.


## Load definitions without running upstream experiments

In [ ]:
from pathlib import Path
import nbformat
PROJECT = Path.cwd().resolve()
while not (PROJECT / 'AGENTS.md').is_file() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
for cell in nbformat.read(PROJECT / 'experiments/ahr_v2/00_contracts_and_recovery.ipynb', 4).cells:
    if cell.cell_type == 'code' and 'ahr-export' in cell.metadata.get('tags', []):
        exec(compile(cell.source, '00:ahr-export', 'exec'), globals())


## Detached worker and control definitions

In [ ]:
import shlex, subprocess, traceback
from zoneinfo import ZoneInfo

GO2_SEQUENCE = (
    '00_contracts_and_recovery.ipynb', '01_rules_and_replay.ipynb',
    '02_scout_worker_network.ipynb', '03_batched_puct_search.ipynb',
    '04_selfplay_and_training.ipynb', '05_evaluation_and_progress.ipynb',
    '06_play_and_watch.ipynb', '07_validate_and_launch.ipynb',
)

def go2_control_sources(project):
    project = Path(project)
    return {GO2_ROOT+'/'+name: go2_code_sha(project/GO2_ROOT/name) for name in GO2_SEQUENCE}

def go2_control_imports(project, include_evaluation=True):
    project=Path(project)
    namespace={'__name__':'ahr_v2_control_exports'}
    for name in GO2_SEQUENCE[:6 if include_evaluation else 5]:
        go2_import_exports(project/GO2_ROOT/name, namespace)
    return namespace

def go2_dublin_time(iso_timestamp):
    return dt.datetime.fromisoformat(iso_timestamp).astimezone(ZoneInfo('Europe/Dublin')).strftime('%Y-%m-%d %H:%M %Z')

def go2_control_read(project, path):
    return json.loads((Path(project)/path).read_text())

def go2_longrun_preflight(project, config):
    project=Path(project); sources=go2_control_sources(project)
    foundation=go2_source_identity(project)
    checks={}
    for name in ('checkpoint','rules','network','search'):
        receipt=go2_control_read(project, 'reports/ahr_v2/'+name+'_validation.json')
        checks[name] = receipt.get('status') == 'PASS' and receipt.get('sources') == foundation
    checks['tmux'] = subprocess.run(['tmux','-V'],capture_output=True,text=True).returncode==0
    checks['gpu'] = torch.cuda.is_available() and config['device'].startswith('cuda:') and int(config['device'].split(':')[1]) < torch.cuda.device_count()
    checks['gpu_isolation'] = config['device']=='cuda:1'
    checks['pure_ahr_learning'] = config.get('training_source') == 'ahr_selfplay'
    training_receipt=go2_control_read(project, 'reports/ahr_v2/training_validation.json')
    checks['training_recovery'] = training_receipt.get('status') == 'PASS'
    checks['cuda_training_recovery'] = any(f.get('device')=='cuda:1' and f.get('fresh_kernel_exact_scientific_state') is True for f in training_receipt.get('fixtures',[]))
    checks['training_source_binding'] = training_receipt.get('sources',{}).get(GO2_ROOT+'/04_selfplay_and_training.ipynb') == sources[GO2_ROOT+'/04_selfplay_and_training.ipynb']
    for name in ('evaluation','play'):
        receipt=go2_control_read(project, 'reports/ahr_v2/'+name+'_validation.json')
        checks[name+'_validation'] = receipt.get('status')=='PASS' and all(sources.get(path)==digest for path,digest in receipt.get('sources',{}).items())
    checks['free_storage'] = os.statvfs(project).f_bavail*os.statvfs(project).f_frsize > 20*1024**3
    if not all(checks.values()):
        raise RuntimeError('Preflight failed: '+repr(checks))
    return {'status':'PASS','checks':checks,'sources':sources,'config_sha256':go2_digest(config),'checked_at_utc':go2_now()}

def go2_review_checkpoint(project, config, namespace, control, final=False):
    # Development matches do not supply training labels and do not change training RNG.
    project=Path(project);run_id=config['run_id']
    snapshot,_=go2_load_checkpoint(project/'checkpoints'/run_id)
    if snapshot is None or int(snapshot.get('metrics',{}).get('optimizer_step',0))<=0:
        return control
    step=int(snapshot['metrics']['optimizer_step'])
    eval_id=run_id+('-final-dev-' if final else '-periodic-dev-')+str(step)
    review_config=namespace['Go2EvalConfig'](games=8 if final else 4,simulations=32,batch_size=8,
        max_plies=400,game_time_seconds=180,move_time_seconds=2)
    control['active_review']={'eval_id':eval_id,'started_at_utc':go2_now(),'status':'RUNNING'}
    go2_atomic_json(project/'reports/ahr_v2'/run_id/'control.json',control)
    review={'eval_id':eval_id,'optimizer_step':step,'status':'PASS',
            'report_path':'reports/ahr_v2/evaluations/'+eval_id+'/report.json','final_development_review':bool(final)}
    rng=go2_capture_rng(cuda_devices=(1,))
    try:
        namespace['go2_evaluate'](project,run_id,eval_id,review_config,which='latest',device='cpu')
    except Exception as error:
        # A diagnostic failure is recorded separately; it must not erase valid learning progress.
        review.update(status='ERROR',error=str(error),traceback=traceback.format_exc())
    finally:
        go2_restore_rng(rng)
    review['finished_at_utc']=go2_now()
    control['active_review']=None
    control.setdefault('reviews',[]).append(review)
    control['last_review_at_utc']=review['finished_at_utc']
    go2_atomic_json(project/'reports/ahr_v2'/run_id/'control.json',control)
    return control

def go2_pipeline(project, config):
    project=Path(project);namespace=go2_control_imports(project);run_id=config['run_id']
    path=project/'reports/ahr_v2'/run_id/'control.json'
    control=json.loads(path.read_text()) if path.is_file() else {'run_id':run_id,'reviews':[],'last_review_at_utc':go2_now()}
    go2_atomic_json(path,control)
    def on_checkpoint(state,entry):
        nonlocal control
        last=dt.datetime.fromisoformat(control['last_review_at_utc'])
        due=(dt.datetime.now(dt.timezone.utc)-last).total_seconds()>=7200
        if due and int(state.get('metrics',{}).get('optimizer_step',0))>0 and not (project/'checkpoints'/run_id/'STOP').exists():
            control=go2_review_checkpoint(project,config,namespace,control)
    result=namespace['go2_run_training'](project,config,on_checkpoint=on_checkpoint)
    stop_file=project/'checkpoints'/run_id/'STOP'
    if not stop_file.exists() and not any(r.get('final_development_review') for r in control['reviews']):
        control=go2_review_checkpoint(project,config,namespace,control,final=True)
    return result

def go2_worker_entry(project, config_path, launch_path):
    project=Path(project);launch=json.loads(Path(launch_path).read_text());config=json.loads(Path(config_path).read_text())
    if go2_control_sources(project)!=launch['sources'] or go2_digest(config)!=launch['config_sha256']:
        raise RuntimeError('Pinned worker source/config identity changed; refusing launch or resume.')
    worker_state=project/'reports/ahr_v2'/config['run_id']/'worker.json'
    start={'status':'RUNNING','pid':os.getpid(),'started_at_utc':go2_now(),'run_id':config['run_id'],'session':launch['session']}
    go2_atomic_json(worker_state,start)
    try:
        result=go2_pipeline(project,config)
    except BaseException as error:
        go2_atomic_json(worker_state,{**start,'status':'ERROR','finished_at_utc':go2_now(),'error':str(error),'traceback':traceback.format_exc()})
        raise
    else:
        go2_atomic_json(worker_state,{**start,'status':'RETURNED','finished_at_utc':go2_now()})
        return result

def go2_launch_training(project, config_path='config/ahr_v2_training.json', session='ahr-v2-selfplay-12h', resume=False):
    project=Path(project).resolve(); config_path=project/config_path; config=json.loads(config_path.read_text())
    preflight=go2_longrun_preflight(project,config)
    if subprocess.run(['tmux','has-session','-t',session],capture_output=True).returncode==0:
        raise RuntimeError('tmux session already exists: '+session+'; attach or inspect progress.')
    directory=project/'checkpoints'/config['run_id']
    if directory.exists():
        # Validate resumable state before starting any worker; never reset it.
        saved,entry=go2_load_checkpoint(directory)
        if saved is not None and not resume:
            raise RuntimeError('Run already has a checkpoint. Use resume=True with its exact configuration.')
    report_dir=project/'reports/ahr_v2'/config['run_id']; report_dir.mkdir(parents=True,exist_ok=True)
    launch_path=report_dir/'launch.json'; runner_path=report_dir/'detached_worker.ipynb'; log_path=report_dir/'tmux.log'
    launch={**preflight,'run_id':config['run_id'],'session':session,'config_path':str(config_path.relative_to(project)),
            'worker_notebook':str(runner_path.relative_to(project)),'log_path':str(log_path.relative_to(project)),
            'launched_at_utc':go2_now(),'requested_hours':config.get('time_budget_hours',12),
            'resume':bool(resume),'attach_command':'tmux attach -t '+session,'detach_keys':'Ctrl-b, then d',
            'checkpoint_path':'checkpoints/'+config['run_id'],'play_notebook':GO2_ROOT+'/06_play_and_watch.ipynb'}
    runner=nbformat.v4.new_notebook(metadata={'kernelspec':{'name':'ahr-go','language':'python','display_name':'AHR Go'}})
    setup_lines=[
        'from pathlib import Path', 'import nbformat', 'PROJECT=Path('+repr(str(project))+')',
        'for name in ["00_contracts_and_recovery.ipynb","07_validate_and_launch.ipynb"]:',
        '    for cell in nbformat.read(PROJECT/"experiments/ahr_v2"/name,4).cells:',
        '        if cell.cell_type=="code" and "ahr-export" in cell.metadata.get("tags",[]): exec(compile(cell.source,name+":ahr-export","exec"),globals())',
        'result=go2_worker_entry(PROJECT,Path('+repr(str(config_path))+'),Path('+repr(str(launch_path))+'))',
        'print("Detached run returned; inspect status and worker.json for its completion reason.")',
    ]
    runner.cells=[nbformat.v4.new_markdown_cell('# Detached AHR v2 worker — generated by notebook 07. Imports definition cells only and resumes the same pinned training run.'),
                  nbformat.v4.new_code_cell(chr(10).join(setup_lines))]
    go2_atomic_bytes(runner_path,nbformat.writes(runner).encode())
    go2_atomic_json(launch_path,launch)
    python=str(project/'.venv/bin/python')
    command=' '.join(shlex.quote(x) for x in [python,'-m','jupyter','nbconvert','--execute','--to','notebook','--inplace','--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=-1',str(runner_path)])+' >> '+shlex.quote(str(log_path))+' 2>&1'
    subprocess.run(['tmux','new-session','-d','-s',session,'-c',str(project),command],check=True,capture_output=True,text=True)
    go2_atomic_json(project/'reports/ahr_v2/active_run.json',launch)
    go2_atomic_json(project/'config/ahr_v2_active_run.json',launch)
    return launch


## Inspect readiness and the active run

In [ ]:
from IPython.display import HTML, display
config_file=PROJECT/'config/ahr_v2_training.json'
if config_file.is_file():
    TRAINING_CONFIG=json.loads(config_file.read_text())
    PREFLIGHT=go2_longrun_preflight(PROJECT,TRAINING_CONFIG)
    display(HTML('<h3>Long-run preflight: '+PREFLIGHT['status']+'</h3><p>GPU 1 · 9×9 · pure AHR self-play · detached tmux</p>'))
    display({key:value for key,value in PREFLIGHT['checks'].items()})
else:
    TRAINING_CONFIG=None
    display(HTML('<p>The launch configuration has not been frozen yet. Notebook 04 validation must pass before launch.</p>'))
active_file=PROJECT/'reports/ahr_v2/active_run.json'
if active_file.is_file():
    ACTIVE_RUN=json.loads(active_file.read_text())
    display({**{key:ACTIVE_RUN.get(key) for key in ['run_id','session','requested_hours','checkpoint_path','attach_command','detach_keys']}, 'launched_at_dublin':go2_dublin_time(ACTIVE_RUN['launched_at_utc'])})
else:
    ACTIVE_RUN=None
    print('No detached AHR v2 run has been launched yet.')


## Start or resume explicitly

Run All performs readiness inspection. The authorized launch is made once with `go2_launch_training(PROJECT)`. This avoids accidentally creating a second writer whenever you open a progress or play notebook.

To attach: `tmux attach -t ahr-v2-selfplay-12h`. Detach with **Ctrl-b**, then **d**. Open notebook **05** for a friendly progress view and **06** to play.

To stop cleanly, create the empty marker `checkpoints/<run_id>/STOP` or use the training notebook's stop control so the next atomic search or optimizer boundary checkpoints the active state. To resume, clear that stop request, keep the exact configuration, and call `go2_launch_training(PROJECT, resume=True)` after the old session has ended. A live session must be inspected instead of relaunched.

The run ends on its active compute budget; improvements and a 50% win rate against GNU Go are evaluation outcomes, not promised results.


In [ ]:
# The launch function is intentionally not called by Run All.
# Initial launch (already authorized): launch = go2_launch_training(PROJECT)
# Exact resume after a clean stop: launch = go2_launch_training(PROJECT, resume=True)
print('Control notebook ready. Use the recorded session and configuration for the same run.')
